# 40 · Outliers na tabela master

Diagnóstico antes da segmentação com k-means. O k-means usa distância entre pontos, então um valor
extremo puxa o centro de um grupo ou vira um grupo sozinho. Este notebook só identifica: nenhuma
linha é alterada ou removida.

Para cada coluna numérica:
- **faixa**: mínimo, percentis e máximo;
- **regra do IQR**: fora de Q1 − 1,5·IQR a Q3 + 1,5·IQR (moderado) e fora de 3·IQR (extremo);
- **z robusto**: distância da mediana em desvios (MAD), acima de 3,5;
- **assimetria** e **% de zeros**: uma coluna concentrada em zero faz a regra do IQR marcar todo valor
  positivo, o que não é outlier, é formato da distribuição.

Depois: valores impossíveis pelas regras do negócio e os casos mais extremos de cada coluna.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

pd.set_option("display.width", 260)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")
P = Path("../dados/processados"); S = Path("../saidas")

m = pd.read_parquet(P / "tabela_master.parquet")
print("tabela master:", m.shape)
num = [c for c in m.columns if pd.api.types.is_numeric_dtype(m[c]) and not pd.api.types.is_bool_dtype(m[c])]
print("colunas numéricas:", len(num)); print(num)
at = m[m.ativa_no_corte]
uso = m[m.dias_historico_treinos == 90]
print("recortes: todas =", len(m), "| com janela de 90 dias completa =", len(uso), "| assinantes ativos =", len(at))

tabela master: (63929, 80)
colunas numéricas: 32
['ano_nascimento', 'idade_no_corte', 'duracao_dias', 'meses_de_casa', 'dias_historico_treinos', 'n_tipos', 'peso_tipo_principal', 'treinos_corrida', 'treinos_funcional', 'treinos_caminhada', 'treinos_forca', 'treinos_yoga', 'treinos_pedal', 'pct_treinos_relogio', 'dias_historico_posts', 'posts_90d', 'n_seguidores', 'n_seguindo', 'n_mutuas', 'treinos_acompanhados_90d', 'dias_sem_treinar', 'treinos_7d', 'treinos_30d', 'treinos_90d', 'treinos_por_semana', 'dias_ativos_90d', 'minutos_90d', 'dias_por_semana_declarado', 'aderencia_meta', 'n_cobrancas_aprovadas', 'receita_liquida', 'n_chamados']
recortes: todas = 63929 | com janela de 90 dias completa = 51684 | assinantes ativos = 14753


## 1. Faixa e regras de outlier por coluna

In [2]:
def diag(d, cols):
    linhas = []
    for c in cols:
        s = d[c].dropna().astype(float)
        if len(s) == 0: continue
        q1, q3 = s.quantile([.25, .75]); iqr = q3 - q1
        med = s.median(); mad = (s - med).abs().median()
        zr = 0.6745 * (s - med) / mad if mad > 0 else pd.Series(np.nan, index=s.index)
        linhas.append({"coluna": c, "n": len(s), "% zeros": (s == 0).mean() * 100, "min": s.min(), "p1": s.quantile(.01), "p25": q1, "mediana": med, "p75": q3,
                       "p99": s.quantile(.99), "max": s.max(), "assimetria": s.skew(),
                       "lim_sup_1,5": q3 + 1.5 * iqr, "% acima 1,5·IQR": (s > q3 + 1.5 * iqr).mean() * 100, "% abaixo 1,5·IQR": (s < q1 - 1.5 * iqr).mean() * 100,
                       "% fora 3·IQR": ((s > q3 + 3 * iqr) | (s < q1 - 3 * iqr)).mean() * 100, "n fora 3·IQR": int(((s > q3 + 3 * iqr) | (s < q1 - 3 * iqr)).sum()),
                       "% z robusto > 3,5": (zr.abs() > 3.5).mean() * 100 if mad > 0 else np.nan, "max ÷ p99": s.max() / s.quantile(.99) if s.quantile(.99) > 0 else np.nan})
    return pd.DataFrame(linhas).set_index("coluna")
d_at = diag(at, num)
print("== ASSINANTES ATIVOS (", len(at), ")"); print(d_at[["n", "% zeros", "min", "p1", "mediana", "p99", "max", "assimetria"]]); print()
print(d_at[["lim_sup_1,5", "% acima 1,5·IQR", "% abaixo 1,5·IQR", "% fora 3·IQR", "n fora 3·IQR", "% z robusto > 3,5", "max ÷ p99"]])

== ASSINANTES ATIVOS ( 14753 )
                               n  % zeros      min       p1  mediana      p99       max  assimetria
coluna                                                                                             
ano_nascimento             13856     0.00 1,952.00 1,953.00 1,979.00 2,000.00  2,007.00       -0.35
idade_no_corte             13856     0.00    18.00    26.00    46.00    73.00     74.00        0.34
duracao_dias               14753     0.22     0.00     3.00   263.00 2,125.96  2,213.00        2.13
meses_de_casa              14753     0.40     0.00     0.10     8.60    69.85     72.70        2.13
dias_historico_treinos     14753     0.00    90.00    90.00    90.00    90.00     90.00        0.00
n_tipos                    14753     5.11     0.00     0.00     4.00     5.00      5.00       -1.16
peso_tipo_principal        13999     0.00    20.00    30.00    65.20   100.00    100.00        0.15
treinos_corrida            14753    48.11     0.00     0.00     1.00 

In [3]:
d_uso = diag(uso, num)
print("== TODAS AS CONTAS COM JANELA DE 90 DIAS COMPLETA (", len(uso), ")"); print(d_uso[["n", "% zeros", "min", "p1", "mediana", "p99", "max", "assimetria", "% acima 1,5·IQR", "% fora 3·IQR", "max ÷ p99"]])
pd.concat({"assinantes ativos": d_at, "janela de 90 dias completa": d_uso}, names=["recorte"]).to_csv(S / "tabela_master_outliers.csv", encoding="utf-8-sig")

== TODAS AS CONTAS COM JANELA DE 90 DIAS COMPLETA ( 51684 )


                               n  % zeros      min       p1  mediana      p99       max  assimetria  % acima 1,5·IQR  % fora 3·IQR  max ÷ p99
coluna                                                                                                                                       
ano_nascimento             48628     0.00 1,952.00 1,953.00 1,979.00 2,000.00  2,008.00       -0.37             0.00          0.00       1.00
idade_no_corte             48628     0.00    17.00    26.00    46.00    73.00     74.00        0.37             0.00          0.00       1.01
duracao_dias               20338     0.16     0.00     5.00   263.00 2,095.00  2,213.00        2.26             7.60          4.11       1.06
meses_de_casa              20338     0.29     0.00     0.20     8.60    68.80     72.70        2.26             7.58          4.08       1.06
dias_historico_treinos     51684     0.00    90.00    90.00    90.00    90.00     90.00        0.00             0.00          0.00       1.00
n_tipo

## 2. Classificação: o que é outlier de verdade e o que é formato da distribuição

In [4]:
def classe(r):
    if r["% zeros"] >= 50: return "concentrada em zero"
    if r["max ÷ p99"] >= 3 or r["% fora 3·IQR"] >= 1: return "cauda longa com extremos"
    if abs(r["assimetria"]) >= 1: return "assimétrica"
    return "bem comportada"
r = d_at.assign(classe=d_at.apply(classe, axis=1))
for k, g in r.groupby("classe"):
    print(f"== {k} ({len(g)})"); print(g[["% zeros", "mediana", "p99", "max", "assimetria", "% fora 3·IQR", "max ÷ p99"]].sort_values("max ÷ p99", ascending=False)); print()

== assimétrica (8)
                   % zeros  mediana      p99       max  assimetria  % fora 3·IQR  max ÷ p99
coluna                                                                                     
minutos_90d           5.14 1,020.00 4,250.96 11,559.00        1.70          0.45       2.72
n_seguidores         25.49     1.00     6.00     13.00        1.10          0.13       2.17
treinos_funcional    20.12     3.00    28.00     60.00        1.94          0.73       2.14
n_mutuas             34.18     1.00     5.00     10.00        1.18          0.01       2.00
treinos_7d           27.55     2.00     9.00     17.00        1.28          0.12       1.89
treinos_30d          11.51     9.00    33.00     62.00        1.05          0.14       1.88
posts_90d            26.84     1.00     6.00      9.00        1.05          0.01       1.50
n_tipos               5.11     4.00     5.00      5.00       -1.16          0.00       1.00

== bem comportada (11)
                           % zeros  m

## 3. Valores impossíveis ou suspeitos pelas regras do negócio

In [5]:
REGRAS = {
 "idade fora de 18 a 75 anos": m.idade_no_corte.notna() & ~m.idade_no_corte.between(18, 75),
 "ano de nascimento antes de 1950 ou depois de 2008": m.ano_nascimento.notna() & ~m.ano_nascimento.between(1950, 2008),
 "data_fim antes do início da assinatura": m.data_fim.notna() & (m.data_fim < m.data_inicio_assinatura),
 "início da assinatura antes do cadastro": m.data_inicio_assinatura.notna() & (m.data_inicio_assinatura < m.data_cadastro),
 "duracao_dias negativa": m.duracao_dias.notna() & (m.duracao_dias < 0),
 "duracao_dias igual a 0": m.duracao_dias.notna() & (m.duracao_dias == 0),
 "receita_liquida negativa": m.receita_liquida.notna() & (m.receita_liquida < 0),
 "assinante sem nenhuma cobrança aprovada": m.ja_assinou & (m.n_cobrancas_aprovadas == 0),
 "mais de 1 treino por dia ativo em média (treinos_90d > 2 × dias_ativos_90d)": m.treinos_90d.notna() & (m.treinos_90d > 2 * m.dias_ativos_90d),
 "dias_ativos_90d acima de 90": m.dias_ativos_90d.notna() & (m.dias_ativos_90d > 90),
 "treinos_30d maior que treinos_90d": m.treinos_30d.notna() & (m.treinos_30d > m.treinos_90d),
 "treinos_7d maior que treinos_30d": m.treinos_7d.notna() & m.treinos_30d.notna() & (m.treinos_7d > m.treinos_30d),
 "média acima de 180 min por treino": m.treinos_90d.notna() & (m.treinos_90d > 0) & (m.minutos_90d / m.treinos_90d > 180),
 "média abaixo de 5 min por treino": m.treinos_90d.notna() & (m.treinos_90d > 0) & (m.minutos_90d / m.treinos_90d < 5),
 "peso_tipo_principal fora de 0 a 100": m.peso_tipo_principal.notna() & ~m.peso_tipo_principal.between(0, 100),
 "pct_treinos_relogio fora de 0 a 100": m.pct_treinos_relogio.notna() & ~m.pct_treinos_relogio.between(0, 100),
 "treino de relógio em conta sem relógio no cadastro": (m.relogio == "nenhum") & (m.pct_treinos_relogio > 0),
 "mais posts do que treinos em 90 dias": m.posts_90d.notna() & m.treinos_90d.notna() & (m.posts_90d > m.treinos_90d),
 "n_mutuas maior que n_seguindo ou n_seguidores": (m.n_mutuas > m.n_seguindo) | (m.n_mutuas > m.n_seguidores),
 "aderencia_meta acima de 3 (treina 3× o que declarou)": m.aderencia_meta.notna() & (m.aderencia_meta > 3),
 "dias_sem_treinar negativo": m.dias_sem_treinar.notna() & (m.dias_sem_treinar < 0),
 "nível alto com treinos_7d = 0": (m.nivel_engajamento == "alto") & (m.treinos_7d == 0),
 "5 ou mais chamados de suporte": m.n_chamados >= 5,
}
t = pd.DataFrame({"contas": {k: int(v.sum()) for k, v in REGRAS.items()}})
t["entre ativos"] = [int((v & m.ativa_no_corte).sum()) for v in REGRAS.values()]
t["% das contas"] = t.contas / len(m) * 100
print(t)

                                                    contas  entre ativos  % das contas
idade fora de 18 a 75 anos                               1             0          0.00
ano de nascimento antes de 1950 ou depois de 2008        0             0          0.00
data_fim antes do início da assinatura                   0             0          0.00
início da assinatura antes do cadastro                   0             0          0.00
duracao_dias negativa                                    0             0          0.00
duracao_dias igual a 0                                  32            32          0.05
receita_liquida negativa                                 0             0          0.00
assinante sem nenhuma cobrança aprovada                847             0          1.32
mais de 1 treino por dia ativo em média (treino...      24             7          0.04
dias_ativos_90d acima de 90                              0             0          0.00
treinos_30d maior que treinos_90d          

In [6]:
print("detalhe das regras que acusaram algo")
for k, v in REGRAS.items():
    if v.sum() == 0: continue
    cols = ["user_id", "situacao", "plano", "data_cadastro", "data_inicio_assinatura", "data_fim", "duracao_dias", "n_cobrancas_aprovadas", "receita_liquida", "treinos_90d", "dias_ativos_90d", "minutos_90d", "posts_90d", "dias_por_semana_declarado", "treinos_por_semana", "aderencia_meta", "idade_no_corte", "n_chamados"]
    print(f"-- {k}: {int(v.sum())} contas"); print(m.loc[v, cols].head(5).to_string()); print()

detalhe das regras que acusaram algo
-- idade fora de 18 a 75 anos: 1 contas
          user_id         situacao plano data_cadastro data_inicio_assinatura data_fim  duracao_dias  n_cobrancas_aprovadas  receita_liquida  treinos_90d  dias_ativos_90d  minutos_90d  posts_90d  dias_por_semana_declarado  treinos_por_semana  aderencia_meta  idade_no_corte  n_chamados
59296  u_ed5b4585  em teste grátis  None    2026-09-28                    NaT      NaT          <NA>                   <NA>              NaN            0                0            0          0                          4                0.00            0.00              17           0

-- duracao_dias igual a 0: 32 contas
          user_id         situacao   plano data_cadastro data_inicio_assinatura data_fim  duracao_dias  n_cobrancas_aprovadas  receita_liquida  treinos_90d  dias_ativos_90d  minutos_90d  posts_90d  dias_por_semana_declarado  treinos_por_semana  aderencia_meta  idade_no_corte  n_chamados
2731   u_0a980c4d  assina

## 4. Os casos mais extremos das colunas de cauda longa

In [7]:
for c in ["treinos_90d", "minutos_90d", "dias_ativos_90d", "treinos_pedal", "treinos_corrida", "posts_90d", "treinos_acompanhados_90d", "n_seguidores", "receita_liquida", "n_cobrancas_aprovadas", "duracao_dias", "aderencia_meta", "dias_sem_treinar", "n_chamados"]:
    s = at[c].dropna().astype(float)
    top = s.sort_values(ascending=False).head(5).round(1).tolist()
    print(f"{c:28s} p99 = {s.quantile(.99):>9,.1f} | 5 maiores entre ativos: {top}")
print()
print("treinos por dia ativo (treinos_90d ÷ dias_ativos_90d), ativos com treino:")
x = (at.treinos_90d / at.dias_ativos_90d).dropna().astype(float); print(x.describe(percentiles=[.5, .9, .99]).round(2).to_dict())
print("minutos por treino:"); x = (at.minutos_90d / at.treinos_90d).dropna().astype(float); print(x.describe(percentiles=[.01, .5, .99]).round(1).to_dict())

treinos_90d                  p99 =      84.0 | 5 maiores entre ativos: [160.0, 131.0, 131.0, 130.0, 128.0]
minutos_90d                  p99 =   4,251.0 | 5 maiores entre ativos: [11559.0, 10528.0, 10160.0, 9292.0, 9108.0]
dias_ativos_90d              p99 =      55.0 | 5 maiores entre ativos: [73.0, 73.0, 72.0, 71.0, 71.0]
treinos_pedal                p99 =      37.0 | 5 maiores entre ativos: [90.0, 85.0, 79.0, 78.0, 73.0]
treinos_corrida              p99 =      52.0 | 5 maiores entre ativos: [100.0, 95.0, 93.0, 90.0, 90.0]
posts_90d                    p99 =       6.0 | 5 maiores entre ativos: [9.0, 9.0, 8.0, 8.0, 8.0]
treinos_acompanhados_90d     p99 =       6.0 | 5 maiores entre ativos: [24.0, 20.0, 19.0, 18.0, 18.0]
n_seguidores                 p99 =       6.0 | 5 maiores entre ativos: [13.0, 11.0, 11.0, 11.0, 11.0]
receita_liquida              p99 =     933.1 | 5 maiores entre ativos: [1007.1, 1007.1, 1007.1, 1007.1, 1007.1]
n_cobrancas_aprovadas        p99 =      28.0 | 5 maiores e

## 5. Contas extremas em várias colunas ao mesmo tempo

In [8]:
# z robusto por coluna entre os ativos, só nas colunas contínuas de uso e comportamento
cols = ["treinos_90d", "dias_ativos_90d", "minutos_90d", "n_tipos", "posts_90d", "treinos_acompanhados_90d", "n_mutuas", "n_seguidores", "duracao_dias", "receita_liquida", "aderencia_meta", "dias_sem_treinar", "n_chamados"]
z = pd.DataFrame(index=at.index)
for c in cols:
    s = at[c].astype(float); q1, q3 = s.quantile([.25, .75]); iqr = q3 - q1
    z[c] = (s > q3 + 3 * iqr) | (s < q1 - 3 * iqr) if iqr > 0 else False
z = z.fillna(False)
k = z.sum(axis=1)
print("ativos por número de colunas em que são extremos (3·IQR):"); print(k.value_counts().sort_index().to_frame("ativos").assign(pct=lambda d: d.ativos / d.ativos.sum() * 100))
print(); print("colunas que mais marcam extremos:"); print(z.sum().sort_values(ascending=False).to_frame("ativos extremos"))
print(); print("contas extremas em 3 ou mais colunas:", int((k >= 3).sum()))
print(at.loc[k >= 3, ["user_id", "plano"] + cols].head(8).to_string())

ativos por número de colunas em que são extremos (3·IQR):
   ativos   pct
0   12604 85.43
1    2087 14.15
2      60  0.41
3       2  0.01

colunas que mais marcam extremos:


                          ativos extremos
dias_sem_treinar                     1557
duracao_dias                          530
minutos_90d                            66
aderencia_meta                         22
n_seguidores                           19
treinos_90d                            15
n_mutuas                                2
posts_90d                               2
dias_ativos_90d                         0
n_tipos                                 0
treinos_acompanhados_90d                0
receita_liquida                         0
n_chamados                              0

contas extremas em 3 ou mais colunas: 2
          user_id   plano  treinos_90d  dias_ativos_90d  minutos_90d  n_tipos  posts_90d  treinos_acompanhados_90d  n_mutuas  n_seguidores  duracao_dias  receita_liquida  aderencia_meta  dias_sem_treinar  n_chamados
9747   u_26ec2fcc  mensal           38               29         1569        4          1                        14        10            13          2170   